In [1]:
import pandas as pd
import tqdm, os
import pickle
import warnings
warnings.simplefilter("ignore")
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Subset

import sys
sys.path.insert(0, "../../..")

from utils.acoustics_processing import get_mfccs
from utils.articulation_processing import get_aemri_articulatory_features

In [2]:
df = pd.read_csv("../dataset_construction/datasets/processed_aemri.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[0])

In [3]:
df

,phone1_start,phone1_end,phone2_start,phone2_end,phone1,phone2,context,word,word_id,art_f,ac_f,subject
0,2.28270,2.33270,2.33270,2.43270,h,ɝ,h,her,0.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/wav/u...,usctimit
1,2.43270,2.64270,2.64270,2.67270,ɔ,b,ɔ,auburn,1.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/wav/u...,usctimit
2,2.64270,2.67270,2.67270,2.82270,b,ɝ,ɔ_b,auburn,1.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/wav/u...,usctimit
3,2.67270,2.82270,2.82270,2.86270,ɝ,n,ɔ_b_ɝ,auburn,1.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/wav/u...,usctimit
4,2.86270,2.96270,2.96270,3.19270,h,ɛ,h,hair,2.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/wav/u...,usctimit
...,...,...,...,...,...,...,...,...,...,...,...,...
98930,22.56169,22.59172,22.59172,22.63176,m,u,m,MOVIES,36.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/wav/u...,usctimit
98931,22.59172,22.63176,22.63176,22.68181,u,v,m_u,MOVIES,36.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/wav/u...,usctimit
98932,22.63176,22.68181,22.68181,22.88201,v,i,m_u_v,MOVIES,36.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/wav/u...,usctimit
98933,22.68181,22.88201,22.88201,23.12225,i,z,m_u_v_i,MOVIES,36.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/wav/u...,usctimit


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
from utils.models import MDN
import copy

from utils.models import VideoAutoencoder, SequenceAutoencoder

device = "mps" if torch.backends.mps.is_available() else "cpu"

art_model = VideoAutoencoder(
    input_shape=(5, 68, 68),
    latent_dim=1024,
    use_mlp=True,
    grid_sample_cpu_fallback=True,
).to(device)
art_model.load_state_dict(torch.load("../../encoder_training/ckpts/aemri/art_model_ckpt.pth", map_location=device))
art_model.eval()

ac_model = SequenceAutoencoder(
    feature_shape=30,
    latent_dim=1024,
).to(device)
ac_model.load_state_dict(torch.load("../../encoder_training/ckpts/aemri/ac_model_ckpt.pth", map_location=device))
ac_model.eval()

SequenceAutoencoder(
  (enc_conv): Sequential(
    (0): Conv1d(30, 512, kernel_size=(5,), stride=(1,), padding=(2,))
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv1d(512, 256, kernel_size=(5,), stride=(1,), padding=(2,))
    (4): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): Conv1d(256, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (7): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): ReLU()
  )
  (enc_pool): AdaptiveAvgPool1d(output_size=1)
  (fc_enc): Sequential(
    (0): Linear(in_features=128, out_features=4096, bias=True)
    (1): ReLU()
    (2): Linear(in_features=4096, out_features=2048, bias=True)
    (3): ReLU()
    (4): Linear(in_features=2048, out_features=1024, bias=True)
  )
  (fc_dec): Sequential(
    (0): Linear(in_features=1024, out_features=2048, bias=True)
    (1): ReLU()
    (2): Linear(i

In [5]:
df

,phone1_start,phone1_end,phone2_start,phone2_end,phone1,phone2,context,word,word_id,art_f,ac_f,subject
0,2.28270,2.33270,2.33270,2.43270,h,ɝ,h,her,0.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/wav/u...,usctimit
1,2.43270,2.64270,2.64270,2.67270,ɔ,b,ɔ,auburn,1.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/wav/u...,usctimit
2,2.64270,2.67270,2.67270,2.82270,b,ɝ,ɔ_b,auburn,1.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/wav/u...,usctimit
3,2.67270,2.82270,2.82270,2.86270,ɝ,n,ɔ_b_ɝ,auburn,1.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/wav/u...,usctimit
4,2.86270,2.96270,2.96270,3.19270,h,ɛ,h,hair,2.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/wav/u...,usctimit
...,...,...,...,...,...,...,...,...,...,...,...,...
98930,22.56169,22.59172,22.59172,22.63176,m,u,m,MOVIES,36.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/wav/u...,usctimit
98931,22.59172,22.63176,22.63176,22.68181,u,v,m_u,MOVIES,36.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/wav/u...,usctimit
98932,22.63176,22.68181,22.68181,22.88201,v,i,m_u_v,MOVIES,36.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/wav/u...,usctimit
98933,22.68181,22.88201,22.88201,23.12225,i,z,m_u_v_i,MOVIES,36.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/wav/u...,usctimit


In [6]:
def get_vectors(art_f, ac_f, start, end, subject):
    articulatory_features = get_aemri_articulatory_features(
        art_f,
        start,
        end
    )

    if articulatory_features is not None:

        articulatory_features = (
            torch.from_numpy(articulatory_features)
            .float()
            .unsqueeze(0)
            .to(device)
        )

        art_out, art_z = art_model(
            articulatory_features,
            return_latent=True
        )
    else:
        art_out, art_z = None, None

    mfccs = get_mfccs(ac_f, start, end)

    if mfccs is not None:
        mfccs = (
            torch.from_numpy(mfccs.T)
            .float()
            .unsqueeze(0)
            .to(device)
        )

        ac_out, ac_z = ac_model(mfccs)
    else:
        ac_out, ac_z = None, None

    return {
        "articulatory_features": (
            articulatory_features.detach().cpu().numpy()
            if articulatory_features is not None
            else None
        ),
        "art_z": (
            art_z.detach().cpu().numpy()
            if art_z is not None
            else None
        ),
        "mfccs": (
            mfccs.detach().cpu().numpy()
            if mfccs is not None
            else None
        ),
        "ac_z": (
            ac_z.detach().cpu().numpy()
            if ac_z is not None
            else None
        ),
    }

In [7]:
def distance(v1, v2):
    if v1 is None or v2 is None:
        return None
    return np.linalg.norm(v1 - v2)

In [8]:
pbar = tqdm.tqdm(df.iterrows(), total=len(df))

dists = {
    "art_latent_dist": [],
    "art_raw_dist": [],
    "ac_latent_dist": [],
    "ac_raw_dist": [],
}
for idx, data in pbar:
    
    phone1_vectors = get_vectors(data["art_f"], data["ac_f"], data["phone1_start"], data["phone1_end"], data["subject"])
    phone2_vectors = get_vectors(data["art_f"], data["ac_f"], data["phone2_start"], data["phone2_end"], data["subject"])

    art_latent_dist = distance(phone1_vectors["art_z"], phone2_vectors["art_z"])
    art_raw_dist = distance(phone1_vectors["articulatory_features"], phone2_vectors["articulatory_features"])

    ac_latent_dist = distance(phone1_vectors["ac_z"], phone2_vectors["ac_z"])
    ac_raw_dist = distance(phone1_vectors["mfccs"], phone2_vectors["mfccs"])

    dists["art_latent_dist"].append(art_latent_dist)
    dists["art_raw_dist"].append(art_raw_dist)
    dists["ac_latent_dist"].append(ac_latent_dist)
    dists["ac_raw_dist"].append(ac_raw_dist)

pd.DataFrame(dists).to_csv("distances_aemri.csv", index=False)

100%|██████████| 98935/98935 [1:43:14<00:00, 15.97it/s]  
